# Titanic Dataset Analysis - Exploratory Data Analysis Project
## Introduction
The sinking of the RMS Titanic is one of the most infamous shipwrecks in history. On April 15, 1912, during its maiden voyage, the Titanic sank after colliding with an iceberg, resulting in the deaths of more than 1,500 passengers and crew.
In this exploratory data analysis project, we will analyze the Titanic dataset to uncover patterns and insights that might explain the factors that influenced survival rates. Our analysis will focus on several key questions:
- Did social class (Pclass) affect survival rates?
- Were women and children really given priority during evacuation?
- Did the location of a passenger's cabin or their point of embarkation influence their chances of survival?
- How did family relationships impact survival odds?
We'll approach this analysis using Object-Oriented Programming (OOP) principles to create a structured and reusable codebase. By organizing our code into meaningful classes with specific responsibilities, we can make our analysis more modular, maintainable, and easier to understand.
## Setup and Imports

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
from datetime import datetime

# For PDF report generation
from fpdf import FPDF
import base64
from io import BytesIO

# Set style for plots
plt.style.use('seaborn-v0_8')
sns.set_palette('viridis')

# Configure visualizations for better display
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## Data Loading and Initial Exploration
First, we'll create a class to load and explore the dataset. This class will be responsible for reading the data from a CSV file and providing methods to examine its structure and contents.
The `DataLoader` class encapsulates all operations related to loading and exploring the initial dataset, keeping our code organized and making it easier to reuse these functions for other datasets in the future.

In [ ]:
class DataLoader:
    """
    A class for loading and performing initial exploration of a dataset.
    
    This class provides methods to:
    - Load data from a CSV file
    - Display basic information about the dataset structure
    - Examine specific columns in detail
    """
    
    def __init__(self, file_path):
        """
        Initialize with the file path to the dataset.
        
        Parameters:
        -----------
        file_path : str
            Path to the CSV file containing the dataset
        """
        self.file_path = file_path
        self.data = None
        
    def load_data(self):
        """
        Load the dataset from the file path.
        
        Returns:
        --------
        pandas.DataFrame
            The loaded dataset, or None if loading failed
        """
        try:
            self.data = pd.read_csv(self.file_path)
            print(f"Dataset loaded successfully with {self.data.shape[0]} rows and {self.data.shape[1]} columns.")
            return self.data
        except Exception as e:
            print(f"Error loading dataset: {e}")
            return None
    
    def get_basic_info(self):
        """
        Display basic information about the dataset, including:
        - Data types and non-null counts
        - First few rows of data
        - Statistical summary
        - Missing value analysis
        """
        if self.data is None:
            print("No data loaded yet. Please call load_data() first.")
            return
        
        print("\n=== Dataset Info ===")
        print(self.data.info())
        
        print("\n=== First 5 Rows ===")
        display(self.data.head())
        
        print("\n=== Basic Statistics ===")
        display(self.data.describe())
        
        print("\n=== Missing Values ===")
        missing_values = self.data.isnull().sum()
        missing_percent = (missing_values / len(self.data)) * 100
        missing_df = pd.DataFrame({
            'Missing Values': missing_values,
            'Percentage': missing_percent
        })
        display(missing_df[missing_df['Missing Values'] > 0].sort_values('Missing Values', ascending=False))
        
        # Visualize missing values
        plt.figure(figsize=(12, 6))
        missing_df = missing_df[missing_df['Missing Values'] > 0].sort_values('Missing Values', ascending=False)
        
        if not missing_df.empty:
            plt.bar(missing_df.index, missing_df['Percentage'])
            plt.title('Percentage of Missing Values by Column')
            plt.xlabel('Columns')
            plt.ylabel('Percentage Missing')
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
            
            print("\nObservations on Missing Data:")
            for col in missing_df.index:
                percent = missing_df.loc[col, 'Percentage']
                print(f"- {col}: {percent:.1f}% missing values")
                
                if col == 'Age':
                    print("  Age is missing for about 1/5 of passengers, which could affect age-related analysis.")
                elif col == 'Cabin':
                    print("  Cabin has a high proportion of missing values, which limits cabin location analysis.")
                elif col == 'Embarked':
                    print("  Only a few values are missing for Embarked, which shouldn't significantly impact analysis.")
        else:
            print("No missing values found in the dataset!")
        
    def get_column_info(self, column_name):
        """
        Display detailed information about a specific column.
        
        Parameters:
        -----------
        column_name : str
            Name of the column to analyze
        """
        if self.data is None:
            print("No data loaded yet. Please call load_data() first.")
            return
        
        if column_name not in self.data.columns:
            print(f"Column '{column_name}' not found in the dataset.")
            return
        
        column_data = self.data[column_name]
        
        print(f"\n=== Info for Column: {column_name} ===")
        print(f"Data Type: {column_data.dtype}")
        print(f"Number of Unique Values: {column_data.nunique()}")
        print(f"Number of Missing Values: {column_data.isnull().sum()} ({column_data.isnull().sum()/len(column_data)*100:.2f}%)")
        
        if column_data.dtype == 'object':
            print("\nTop 10 Most Frequent Values:")
            display(column_data.value_counts().head(10))
            
            # Visualize categorical distribution
            plt.figure(figsize=(10, 5))
            value_counts = column_data.value_counts()
            
            if len(value_counts) <= 20:  # Only plot if not too many categories
                plt.bar(value_counts.index, value_counts.values)
                plt.title(f'Distribution of {column_name}')
                plt.xlabel(column_name)
                plt.ylabel('Count')
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()
                
                print(f"\nInsights about {column_name}:")
                if column_name == 'Sex':
                    male_percent = column_data.value_counts(normalize=True)['male'] * 100
                    print(f"- Males comprise {male_percent:.1f}% of passengers, females {100-male_percent:.1f}%")
                elif column_name == 'Embarked':
                    most_common = column_data.value_counts().index[0]
                    most_common_pct = column_data.value_counts(normalize=True).iloc[0] * 100
                    print(f"- Most passengers ({most_common_pct:.1f}%) embarked from {most_common}")
                elif column_name == 'Pclass':
                    third_class_pct = column_data.value_counts(normalize=True)[3] * 100 if 3 in column_data.value_counts().index else 0
                    print(f"- Third class passengers make up {third_class_pct:.1f}% of all passengers")
        else:
            print("\nSummary Statistics:")
            display(column_data.describe())
            
            plt.figure(figsize=(12, 5))
            
            plt.subplot(1, 2, 1)
            sns.histplot(column_data.dropna(), kde=True)
            plt.title(f'Distribution of {column_name}')
            
            plt.subplot(1, 2, 2)
            sns.boxplot(y=column_data.dropna())
            plt.title(f'Boxplot of {column_name}')
            
            plt.tight_layout()
            plt.show()
            
            print(f"\nInsights about {column_name}:")
            if column_name == 'Age':
                print(f"- Average age is {column_data.mean():.1f} years")
                children = (column_data < 18).sum()
                children_pct = children / column_data.count() * 100
                print(f"- There were {children} children (<18 years) on board ({children_pct:.1f}% of passengers with known age)")
            elif column_name == 'Fare':
                print(f"- Average fare was £{column_data.mean():.2f}")
                print(f"- There's a wide range in fares (£{column_data.min():.2f} to £{column_data.max():.2f}), suggesting significant class differences")
            elif column_name == 'SibSp':
                pct_with_siblings = (column_data > 0).sum() / len(column_data) * 100
                print(f"- {pct_with_siblings:.1f}% of passengers traveled with siblings or spouses")
            elif column_name == 'Parch':
                pct_with_parents = (column_data > 0).sum() / len(column_data) * 100
                print(f"- {pct_with_parents:.1f}% of passengers traveled with parents or children")

In [ ]:
# Initialize the data loader and load the dataset
loader = DataLoader(r"C:\Users\Evi\Downloads\43.+Titanic-Dataset.csv")
df = loader.load_data()

# Get basic information about the dataset
loader.get_basic_info()

Now let's use the DataLoader class to load and perform an initial exploration of the Titanic dataset:

Let's explore some of the key columns in the dataset to better understand their distributions and patterns:

In [ ]:
# Explore demographic columns
for column in ['Sex', 'Age', 'Pclass']:
    loader.get_column_info(column)

# Explore fare information
loader.get_column_info('Fare')

# Explore family-related columns
for column in ['SibSp', 'Parch']:
    loader.get_column_info(column)

# Explore survival information
loader.get_column_info('Survived')

## Data Cleaning and Preprocessing
Now that we've explored the raw data, we need to clean it and prepare it for analysis. We'll create a DataCleaner class to handle this process.
The data cleaning process will include:
1. Handling missing values in different columns
2. Identifying and treating outliers
3. Creating new features that might provide additional insights
4. Standardizing and transforming variables as needed
By encapsulating these operations in a class, we make our preprocessing steps organized, reusable, and transparent.

In [ ]:
class DataCleaner:
    """
    A class for cleaning and preprocessing the Titanic dataset.
    
    This class provides methods to:
    - Handle missing values in different columns
    - Identify and treat outliers
    - Create new features for analysis
    - Apply a complete data cleaning pipeline
    """
    
    def __init__(self, data):
        """
        Initialize with a pandas DataFrame.
        
        Parameters:
        -----------
        data : pandas.DataFrame
            The dataset to clean and preprocess
        """
        self.data = data.copy()
        self.original_data = data.copy()  # Keep a copy of the original data
        
    def reset_data(self):
        """Reset the data to its original state."""
        self.data = self.original_data.copy()
        print("Data reset to original state.")
        
    def handle_missing_values(self):
        """
        Handle missing values in the dataset using appropriate strategies for each column.
        
        Returns:
        --------
        pandas.DataFrame
            The dataset with missing values handled
        """
        print("Handling missing values using column-specific strategies...")
        
        # Age: Impute with median based on Pclass and Sex
        if 'Age' in self.data.columns and self.data['Age'].isnull().sum() > 0:
            # Create a function to fill missing ages
            def fill_age(row):
                if pd.isnull(row['Age']):
                    condition = (
                        (self.data['Sex'] == row['Sex']) & 
                        (self.data['Pclass'] == row['Pclass'])
                    )
                    median_age = self.data.loc[condition, 'Age'].median()
                    return median_age if not pd.isnull(median_age) else self.data['Age'].median()
                return row['Age']
            
            self.data['Age'] = self.data.apply(fill_age, axis=1)
            print(f"- Imputed {self.original_data['Age'].isnull().sum()} missing Age values")
            print("  Strategy: Using median age for same gender and passenger class")
        
        # Embarked: Fill with most common value (mode)
        if 'Embarked' in self.data.columns and self.data['Embarked'].isnull().sum() > 0:
            mode_embarked = self.data['Embarked'].mode()[0]
            self.data['Embarked'] = self.data['Embarked'].fillna(mode_embarked)
            print(f"- Filled {self.original_data['Embarked'].isnull().sum()} missing Embarked values with mode: {mode_embarked}")
            print("  Strategy: Using the most common embarkation port")
        
        # Fare: Impute with median based on Pclass
        if 'Fare' in self.data.columns and self.data['Fare'].isnull().sum() > 0:
            for pclass in self.data['Pclass'].unique():
                median_fare = self.data[self.data['Pclass'] == pclass]['Fare'].median()
                condition = (self.data['Fare'].isnull()) & (self.data['Pclass'] == pclass)
                self.data.loc[condition, 'Fare'] = median_fare
            print(f"- Imputed {self.original_data['Fare'].isnull().sum()} missing Fare values based on Pclass")
            print("  Strategy: Using median fare for the same passenger class")
        
        # Cabin: We'll create a new feature instead of imputing
        if 'Cabin' in self.data.columns:
            self.data['Has_Cabin'] = self.data['Cabin'].notna().astype(int)
            print("- Created 'Has_Cabin' feature (1 if cabin is known, 0 if not)")
            print("  Strategy: Converting missing cabin information into a meaningful feature")
        
        return self.data
    
    def handle_outliers(self, columns, method='iqr', threshold=1.5):
        """
        Identify and handle outliers in specified columns.
        
        Parameters:
        -----------
        columns : list
            List of column names to check for outliers
        method : str
            Method to use for identifying outliers ('iqr' or 'zscore')
        threshold : float
            Threshold for outlier detection
            
        Returns:
        --------
        pandas.DataFrame
            The dataset with outliers handled
        """
        print(f"Checking for outliers using {method} method with threshold {threshold}...")
        
        for column in columns:
            if column not in self.data.columns:
                print(f"Column '{column}' not found in the dataset.")
                continue
                
            if not np.issubdtype(self.data[column].dtype, np.number):
                print(f"Column '{column}' is not numeric. Skipping.")
                continue
                
            # Create a copy of the column to avoid warnings
            col_data = self.data[column].copy()
            
            # Remove NaN values for outlier detection
            col_data = col_data.dropna()
            
            if method == 'iqr':
                Q1 = col_data.quantile(0.25)
                Q3 = col_data.quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - threshold * IQR
                upper_bound = Q3 + threshold * IQR
                
                outliers = ((col_data < lower_bound) | (col_data > upper_bound)).sum()
                outlier_percent = outliers/len(col_data)*100
                print(f"- {column}: {outliers} outliers detected ({outlier_percent:.2f}%)")
                
                if outliers > 0:
                    # Cap the outliers instead of removing them
                    self.data[column] = self.data[column].clip(lower_bound, upper_bound)
                    print(f"  Action: Capped values to range [{lower_bound:.2f}, {upper_bound:.2f}]")
                
            elif method == 'zscore':
                z_scores = stats.zscore(col_data)
                outliers = (abs(z_scores) > threshold).sum()
                outlier_percent = outliers/len(z_scores)*100
                print(f"- {column}: {outliers} outliers detected ({outlier_percent:.2f}%)")
                
                if outliers > 0:
                    # Cap the outliers
                    self.data.loc[abs(stats.zscore(self.data[column].dropna())) > threshold, column] = \
                        self.data[column].median()
                    print(f"  Action: Replaced outliers with the median value ({self.data[column].median():.2f})")
            
        return self.data
    
    def create_features(self):
        """
        Create new features from existing ones to enhance analysis.
        
        Returns:
        --------
        pandas.DataFrame
            The dataset with new features added
        """
        print("Creating new features...")
        
        # Extract title from name
        if 'Name' in self.data.columns:
            self.data['Title'] = self.data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
            
            # Replace rare titles
            title_mapping = {
                "Mr": "Mr",
                "Miss": "Miss",
                "Mrs": "Mrs",
                "Master": "Master",
                "Dr": "Officer",
                "Rev": "Officer",
                "Col": "Officer",
                "Major": "Officer",
                "Mlle": "Miss",
                "Mme": "Mrs",
                "Don": "Royalty",
                "Sir": "Royalty",
                "Lady": "Royalty",
                "Countess": "Royalty",
                "Jonkheer": "Royalty",
                "Dona": "Royalty",
                "Capt": "Officer",
                "Ms": "Mrs"
            }
            
            self.data['Title'] = self.data['Title'].map(lambda x: title_mapping.get(x, 'Other'))
            print(f"- Created 'Title' feature with {self.data['Title'].nunique()} unique titles")
            print(f"  Title distribution: {self.data['Title'].value_counts().to_dict()}")
            print("  Importance: Titles can capture social status and gender roles beyond simple 'Sex' classification")
        
        # Create family size feature
        if 'SibSp' in self.data.columns and 'Parch' in self.data.columns:
            self.data['FamilySize'] = self.data['SibSp'] + self.data['Parch'] + 1  # +1 for the passenger themselves
            print("- Created 'FamilySize' feature")
            print("  Importance: Family size may have influenced evacuation priority and group survival")
            
            # Create family type feature
            self.data['FamilyType'] = pd.cut(
                self.data['FamilySize'],
                bins=[0, 1, 4, float('inf')],
                labels=['Solo', 'Small', 'Large']
            )
            print("- Created 'FamilyType' feature (Solo, Small, or Large family)")
            print("  Groups: Solo (1), Small family (2-4), Large family (5+)")
        
        # Extract cabin deck
        if 'Cabin' in self.data.columns:
            self.data['Deck'] = self.data['Cabin'].str.slice(0, 1)
            self.data['Deck'] = self.data['Deck'].fillna('U')  # U for Unknown
            print(f"- Created 'Deck' feature with {self.data['Deck'].nunique()} unique values")
            print(f"  Deck distribution: {self.data['Deck'].value_counts().to_dict()}")
            print("  Importance: Cabin deck may indicate proximity to lifeboats and ease of escape")
        
        # Create age groups
        if 'Age' in self.data.columns:
            self.data['AgeGroup'] = pd.cut(
                self.data['Age'],
                bins=[0, 12, 18, 35, 50, 65, float('inf')],
                labels=['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior', 'Elderly']
            )
            print("- Created 'AgeGroup' feature")
            print("  Groups: Child (0-12), Teenager (12-18), Young Adult (18-35), Adult (35-50), Senior (50-65), Elderly (65+)")
            print("  Importance: Age groups help analyze 'women and children first' evacuation policy")
        
        # Create fare categories
        if 'Fare' in self.data.columns:
            self.data['FareCategory'] = pd.qcut(
                self.data['Fare'],
                q=4,
                labels=['Low', 'Medium-Low', 'Medium-High', 'High']
            )
            print("- Created 'FareCategory' feature")
            print("  Categories: Low, Medium-Low, Medium-High, High (based on quartiles)")
            print("  Importance: Fare categories may indicate wealth beyond passenger class")
        
        return self.data
    
    def clean_data(self):
        """
        Perform the full data cleaning pipeline.
        
        Returns:
        --------
        pandas.DataFrame
            The fully cleaned and preprocessed dataset
        """
        print("Starting data cleaning pipeline...")
        
        # Handle missing values
        self.handle_missing_values()
        
        # Handle outliers in numeric columns
        self.handle_outliers(['Age', 'Fare'])
        
        # Create new features
        self.create_features()
        
        print("\nData cleaning complete!")
        return self.data
    print(type(df))

Now let's use the DataCleaner class to clean and preprocess our dataset:

In [ ]:
# Initialize the data cleaner and clean the data
cleaner = DataCleaner(df)
cleaned_df = cleaner.clean_data()

# Display the cleaned dataframe info
print("\nCleaned DataFrame Info:")
print(cleaned_df.info())

# Show a sample of the cleaned data with the new features
print("\nSample of cleaned data with new features:")
display(cleaned_df.head())

## Exploratory Data Analysis (EDA)
Now that we have a clean dataset, we can perform exploratory data analysis to uncover patterns and insights. We'll create an ExploratoryAnalysis class to organize our analysis.
This class will include methods to:
- Analyze the overall survival rate
- Explore relationships between survival and categorical variables
- Explore relationships between survival and numeric variables
- Visualize key relationships in the data
By organizing our analysis in a structured class, we can easily explore different aspects of the data while keeping our code clean and maintainable.

In [ ]:
class ExploratoryAnalysis:
    """
    A class for performing exploratory data analysis on the Titanic dataset.
    
    This class provides methods to:
    - Analyze overall survival rates
    - Explore relationships between survival and categorical variables
    - Explore relationships between survival and numeric variables
    - Visualize key relationships in the data
    """
    
    def __init__(self, data):
        """
        Initialize with a pandas DataFrame.
        
        Parameters:
        -----------
        data : pandas.DataFrame
            The dataset to analyze
        """
        self.data = data
        self.insights = []  # Store key insights for later reporting
        
    def add_insight(self, insight):
        """Add an insight to the collection for reporting."""
        self.insights.append(insight)
        print(f"✓ INSIGHT: {insight}")
        
    def survival_rate_overview(self):
        """
        Display and visualize the overall survival rate.
        
        This method:
        - Calculates the overall survival percentage
        - Creates a bar chart showing the survival distribution
        - Identifies key insights about survival rates
        """
        if 'Survived' not in self.data.columns:
            print("No 'Survived' column found in the dataset.")
            return
        
        survived_count = self.data['Survived'].sum()
        total = len(self.data)
        survival_rate = survived_count / total * 100
        
        print(f"Overall Survival Rate: {survival_rate:.2f}% ({survived_count}/{total})")
        
        # Add insight
        self.add_insight(f"Only {survival_rate:.1f}% of passengers survived the Titanic disaster")
        
        plt.figure(figsize=(8, 6))
        sns.countplot(x='Survived', data=self.data, palette=['#FF5555', '#55AA55'])
        plt.title('Survival Distribution', fontsize=14)
        plt.xlabel('Survived (0 = No, 1 = Yes)', fontsize=12)
        plt.ylabel('Count', fontsize=12)
        plt.xticks([0, 1], ['No', 'Yes'])
        
        # Add percentage labels
        total = len(self.data)
        for i, count in enumerate(self.data['Survived'].value_counts().sort_index()):
            percentage = count / total * 100
            plt.text(i, count + 15, f'{percentage:.1f}%', ha='center')
            
        plt.grid(axis='y', alpha=0.3)
        plt.show()
        
    def survival_by_category(self, category, figsize=(12, 6)):
        """
        Analyze and visualize survival rates by a categorical variable.
        
        Parameters:
        -----------
        category : str
            Name of the categorical variable to analyze
        figsize : tuple
            Figure size for the plot
            
        This method:
        - Calculates survival rates for each category
        - Creates visualizations to compare survival across categories
        - Identifies key insights about survival patterns
        """
        if 'Survived' not in self.data.columns:
            print("No 'Survived' column found in the dataset.")
            return
            
        if category not in self.data.columns:
            print(f"Category '{category}' not found in the dataset.")
            return
        
        # Calculate survival rate by category
        survival_by_cat = self.data.groupby(category)['Survived'].mean().sort_values(ascending=False) * 100
        
        print(f"\n=== Survival Rate by {category} ===")
        for cat, rate in survival_by_cat.items():
            count = len(self.data[self.data[category] == cat])
            print(f"- {cat}: {rate:.2f}% ({count} passengers)")
        
        # Add insights based on the category
        if category == 'Sex':
            female_rate = survival_by_cat.get('female', 0)
            male_rate = survival_by_cat.get('male', 0)
            diff = female_rate - male_rate
            if diff > 40:
                self.add_insight(f"Women had dramatically higher survival rates ({female_rate:.1f}%) than men ({male_rate:.1f}%)")
            else:
                self.add_insight(f"Women had higher survival rates ({female_rate:.1f}%) than men ({male_rate:.1f}%)")
        
        elif category == 'Pclass':
            if 1 in survival_by_cat.index and 3 in survival_by_cat.index:
                first_rate = survival_by_cat[1]
                third_rate = survival_by_cat[3]
                diff = first_rate - third_rate
                self.add_insight(f"First class passengers were {diff/third_rate:.1f}x more likely to survive than third class")
        
        elif category == 'AgeGroup' and 'Child' in survival_by_cat.index:
            child_rate = survival_by_cat['Child']
            adult_rates = [survival_by_cat[group] for group in survival_by_cat.index if group != 'Child']
            avg_adult_rate = sum(adult_rates) / len(adult_rates) if adult_rates else 0
            self.add_insight(f"Children had {child_rate:.1f}% survival rate compared to {avg_adult_rate:.1f}% average for adults")
        
        elif category == 'Title':
            if 'Mrs' in survival_by_cat.index and 'Mr' in survival_by_cat.index:
                self.add_insight(f"Married women (Mrs) had {survival_by_cat['Mrs']:.1f}% survival rate compared to {survival_by_cat['Mr']:.1f}% for adult men")
        
        elif category == 'FamilyType':
            categories = list(survival_by_cat.index)
            if 'Solo' in categories and 'Small' in categories:
                solo_rate = survival_by_cat['Solo']
                small_rate = survival_by_cat['Small']
                self.add_insight(f"Passengers in small families had {small_rate:.1f}% survival rate compared to {solo_rate:.1f}% for solo travelers")
        
        # Create visualizations
        plt.figure(figsize=figsize)
        
        plt.subplot(1, 2, 1)
        ax1 = sns.countplot(x=category, hue='Survived', data=self.data, palette=['#FF5555', '#55AA55'])
        plt.title(f'Survival Count by {category}', fontsize=14)
        plt.xlabel(category, fontsize=12)
        plt.ylabel('Count', fontsize=12)
        plt.legend(['Did not survive', 'Survived'])
        
        # Add count labels
        for p in ax1.patches:
            height = p.get_height()
            if height > 0:
                ax1.text(p.get_x() + p.get_width()/2.,
                        height + 5,
                        f'{int(height)}',
                        ha="center") 
        
        plt.subplot(1, 2, 2)
        ax2 = sns.barplot(x=survival_by_cat.index, y=survival_by_cat.values, palette='viridis')
        plt.title(f'Survival Rate by {category}', fontsize=14)
        plt.xlabel(category, fontsize=12)
        plt.ylabel('Survival Rate (%)', fontsize=12)
        plt.ylim(0, 100)
        
        # Add percentage labels
        for i, rate in enumerate(survival_by_cat):
            plt.text(i, rate + 3, f'{rate:.1f}%', ha='center')
        
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
        
    def survival_by_numeric(self, numeric_col, bins=10, figsize=(12, 6)):
        """
        Analyze and visualize survival rates by a numeric variable.
        
        Parameters:
        -----------
        numeric_col : str
            Name of the numeric variable to analyze
        bins : int
            Number of bins for histogram
        figsize : tuple
            Figure size for the plot
            
        This method:
        - Bins the numeric variable
        - Calculates survival rates for each bin
        - Creates visualizations to show the relationship between the numeric variable and survival
        - Identifies key insights about the patterns
        """
        if 'Survived' not in self.data.columns:
            print("No 'Survived' column found in the dataset.")
            return
            
        if numeric_col not in self.data.columns:
            print(f"Column '{numeric_col}' not found in the dataset.")
            return
            
        if not np.issubdtype(self.data[numeric_col].dtype, np.number):
            print(f"Column '{numeric_col}' is not numeric.")
            return
        
        # Create binned version of the numeric column
        binned_col = f'{numeric_col}_binned'
        self.data[binned_col] = pd.cut(self.data[numeric_col], bins=bins)
        
        # Calculate survival rate by bin
        survival_by_bin = self.data.groupby(binned_col)['Survived'].mean().sort_index() * 100
        counts_by_bin = self.data.groupby(binned_col).size()
        
        print(f"\n=== Survival Rate by {numeric_col} bins ===")
        for bin_range, rate in survival_by_bin.items():
            count = counts_by_bin[bin_range]
            print(f"- {bin_range}: {rate:.2f}% ({count} passengers)")
        
        # Add insights based on the pattern
        if numeric_col == 'Age':
            child_bins = [b for b in survival_by_bin.index if b.left < 12]
            if child_bins:
                child_rates = [survival_by_bin[b] for b in child_bins]
                avg_child_rate = sum(child_rates) / len(child_rates)
                
                adult_bins = [b for b in survival_by_bin.index if b.left >= 18]
                if adult_bins:
                    adult_rates = [survival_by_bin[b] for b in adult_bins]
                    avg_adult_rate = sum(adult_rates) / len(adult_rates)
                    
                    self.add_insight(f"Children under 12 had {avg_child_rate:.1f}% survival rate compared to {avg_adult_rate:.1f}% for adults")
            
            elderly_bins = [b for b in survival_by_bin.index if b.left >= 60]
            if elderly_bins:
                elderly_rates = [survival_by_bin[b] for b in elderly_bins]
                avg_elderly_rate = sum(elderly_rates) / len(elderly_rates)
                
                self.add_insight(f"Elderly passengers (60+) had {avg_elderly_rate:.1f}% survival rate")
        
        elif numeric_col == 'Fare':
            # Check if higher fares had higher survival rates
            low_bin = survival_by_bin.index[0]
            high_bin = survival_by_bin.index[-1]
            low_rate = survival_by_bin[low_bin]
            high_rate = survival_by_bin[high_bin]
            
            if high_rate > low_rate * 2:
                self.add_insight(f"Passengers paying highest fares ({high_bin}) had {high_rate:.1f}% survival rate, more than double the {low_rate:.1f}% rate for lowest fare passengers")
        
        elif numeric_col == 'FamilySize':
            # Compare solo travelers to small families
            solo_bin = None
            small_family_bin = None
            
            for bin_range in survival_by_bin.index:
                if bin_range.left == 0 and bin_range.right == 1:
                    solo_bin = bin_range
                elif bin_range.left == 1 and bin_range.right in [2, 3, 4]:
                    small_family_bin = bin_range
            
            if solo_bin and small_family_bin:
                solo_rate = survival_by_bin[solo_bin]
                small_fam_rate = survival_by_bin[small_family_bin]
                
                if small_fam_rate > solo_rate:
                    self.add_insight(f"Small families (2-4 members) had better survival rates ({small_fam_rate:.1f}%) than solo travelers ({solo_rate:.1f}%)")
                else:
                    self.add_insight(f"Solo travelers had better survival rates ({solo_rate:.1f}%) than small families ({small_fam_rate:.1f}%)")
        
        # Create visualizations
        plt.figure(figsize=figsize)
        
        plt.subplot(1, 2, 1)
        sns.histplot(data=self.data, x=numeric_col, hue='Survived', bins=bins, kde=True, 
                     palette=['#FF5555', '#55AA55'], multiple="stack")
        plt.title(f'Distribution of {numeric_col} by Survival', fontsize=14)
        plt.xlabel(numeric_col, fontsize=12)
        plt.ylabel('Count', fontsize=12)
        plt.legend(['Did not survive', 'Survived'])
        
        plt.subplot(1, 2, 2)
        # Convert bin ranges to strings for better display
        bin_labels = [str(b).replace('(', '').replace(']', '').replace(', ', '-') for b in survival_by_bin.index]
        ax = plt.bar(bin_labels, survival_by_bin.values, color=sns.color_palette('viridis', len(bin_labels)))
        plt.title(f'Survival Rate by {numeric_col}', fontsize=14)
        plt.xlabel(f'{numeric_col} Range', fontsize=12)
        plt.ylabel('Survival Rate (%)', fontsize=12)
        plt.xticks(rotation=45)
        plt.ylim(0, 100)
        
        # Add percentage labels
        for i, rate in enumerate(survival_by_bin.values):
            plt.text(i, rate + 3, f'{rate:.1f}%', ha='center')
        
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Remove the temporary binned column
        self.data = self.data.drop(columns=[binned_col])
        
    def correlation_analysis(self, method='pearson'):
        """
        Analyze correlations between features.
        
        Parameters:
        -----------
        method : str
            Correlation method ('pearson', 'spearman', or 'kendall')
            
        This method:
        - Computes the correlation matrix for numeric variables
        - Visualizes correlations with a heatmap
        - Highlights features most correlated with survival
        - Identifies key insights from correlations
        """
        # Select only numeric columns
        numeric_data = self.data.select_dtypes(include=['int64', 'float64'])
        
        # Compute correlation matrix
        corr_matrix = numeric_data.corr(method=method)
        
        # Plot correlation matrix
        plt.figure(figsize=(12, 10))
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Create mask for upper triangle
        sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', fmt='.2f', 
                    linewidths=0.5, vmin=-1, vmax=1)
        plt.title(f'Correlation Matrix ({method.capitalize()} Method)', fontsize=14)
        plt.tight_layout()
        plt.show()
        
        # Focus on correlations with survival
        if 'Survived' in numeric_data.columns:
            survival_corr = corr_matrix['Survived'].sort_values(ascending=False)
            
            print("\n=== Feature Correlations with Survival ===")
            for feature, corr in survival_corr.items():
                if feature != 'Survived':
                    corr_strength = "Strong" if abs(corr) > 0.5 else "Moderate" if abs(corr) > 0.3 else "Weak"
                    corr_direction = "positive" if corr > 0 else "negative"
                    print(f"- {feature}: {corr:.4f} ({corr_strength} {corr_direction} correlation)")
                    
            # Add insights based on correlations
            top_pos_corr = survival_corr[1:3] if len(survival_corr) > 3 else survival_corr[1:]
            top_neg_corr = survival_corr.tail(2)
            
            for feature, corr in top_pos_corr.items():
                if corr > 0.1:
                    self.add_insight(f"{feature} shows a {corr:.2f} correlation with survival, suggesting it was an important factor")
            
            for feature, corr in top_neg_corr.items():
                if corr < -0.1:
                    self.add_insight(f"{feature} shows a {corr:.2f} correlation with survival, indicating it was associated with lower chances")
                    
            # Visualize top correlations with survival
            plt.figure(figsize=(10, 6))
            top_corr = survival_corr[1:].sort_values(ascending=True)  # Exclude Survived-Survived correlation
            bars = sns.barplot(x=top_corr.values, y=top_corr.index, palette='coolwarm')
            plt.title('Feature Correlations with Survival', fontsize=14)
            plt.xlabel('Correlation Coefficient', fontsize=12)
            plt.axvline(x=0, color='black', linestyle='--')
            
            # Add correlation values
            for i, v in enumerate(top_corr.values):
                bars.text(v + (0.01 if v >= 0 else -0.06), i, f"{v:.2f}", va='center')
            
            plt.tight_layout()
            plt.show()
    
    def analyze_gender_class_survival(self):
        """
        Analyze the combined effect of gender and passenger class on survival.
        
        This method:
        - Creates a cross-tabulation of survival rates by gender and class
        - Visualizes the combined effect with a grouped bar chart
        - Identifies key insights about how these factors interacted
        """
        if 'Sex' not in self.data.columns or 'Pclass' not in self.data.columns:
            print("Required columns 'Sex' and 'Pclass' not found in the dataset.")
            return
        
        # Group by Sex and Pclass
        grouped = self.data.groupby(['Sex', 'Pclass'])['Survived'].mean() * 100
        counts = self.data.groupby(['Sex', 'Pclass']).size()
        
        print("\n=== Survival Rate by Gender and Class ===")
        for (sex, pclass), rate in grouped.items():
            count = counts[(sex, pclass)]
            print(f"- {sex.capitalize()} passengers in class {pclass}: {rate:.2f}% ({count} passengers)")
        
        # Add insights
        female_first = grouped.get(('female', 1), 0)
        female_third = grouped.get(('female', 3), 0)
        male_first = grouped.get(('male', 1), 0)
        male_third = grouped.get(('male', 3), 0)
        
        if female_first > 95 and female_third < 60:
            self.add_insight(f"Almost all first-class women survived ({female_first:.1f}%), while only {female_third:.1f}% of third-class women survived")
        
        if male_first > male_third * 2:
            self.add_insight(f"First-class men had {male_first/male_third:.1f}x better survival odds ({male_first:.1f}%) than third-class men ({male_third:.1f}%)")
        
        # Create a pivot table for plotting
        pivot = pd.pivot_table(
            self.data, 
            values='Survived', 
            index=['Sex'], 
            columns=['Pclass'], 
            aggfunc=np.mean
        ) * 100
        
        # Plot
        plt.figure(figsize=(12, 6))
        pivot.plot(kind='bar', colormap='viridis')
        plt.title('Survival Rate by Gender and Class', fontsize=14)
        plt.xlabel('Gender', fontsize=12)
        plt.ylabel('Survival Rate (%)', fontsize=12)
        plt.xticks(ticks=[0, 1], labels=['Female', 'Male'], rotation=0)
        plt.legend(title='Passenger Class')
        plt.ylim(0, 100)
        
        # Add percentage labels
        for i, (sex, row) in enumerate(pivot.iterrows()):
            for j, rate in enumerate(row):
                plt.text(i - 0.25 + j*0.25, rate + 2, f'{rate:.1f}%', ha='center')
        
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def analyze_age_gender_survival(self):
        """
        Analyze the combined effect of age and gender on survival.
        
        This method:
        - Creates age groups for clearer analysis
        - Calculates survival rates by age group and gender
        - Visualizes these patterns
        - Identifies key insights about age-gender interactions
        """
        if 'Age' not in self.data.columns or 'Sex' not in self.data.columns:
            print("Required columns 'Age' and 'Sex' not found in the dataset.")
            return
        
        # Use AgeGroup if available, otherwise create it
        if 'AgeGroup' not in self.data.columns:
            self.data['AgeGroup'] = pd.cut(
                self.data['Age'],
                bins=[0, 12, 18, 35, 50, 65, float('inf')],
                labels=['Child', 'Teenager', 'Young Adult', 'Adult', 'Senior', 'Elderly']
            )
        
        # Group by AgeGroup and Sex
        grouped = self.data.groupby(['AgeGroup', 'Sex'])['Survived'].mean() * 100
        counts = self.data.groupby(['AgeGroup', 'Sex']).size()
        
        print("\n=== Survival Rate by Age Group and Gender ===")
        for (age_group, sex), rate in grouped.items():
            count = counts[(age_group, sex)]
            print(f"- {sex.capitalize()} {age_group}s: {rate:.2f}% ({count} passengers)")
        
        # Add insights
        child_male = grouped.get(('Child', 'male'), 0)
        child_female = grouped.get(('Child', 'female'), 0)
        adult_male = grouped.get(('Adult', 'male'), 0)
        adult_female = grouped.get(('Adult', 'female'), 0)
        
        if child_male > adult_male and child_male > 50:
            self.add_insight(f"Boy children had much better survival rates ({child_male:.1f}%) than adult men ({adult_male:.1f}%)")
        
        if child_female > 70 and child_male > 50:
            self.add_insight(f"Children of both genders had high survival rates: girls {child_female:.1f}%, boys {child_male:.1f}%")
        
        # Create a pivot table for plotting
        pivot = pd.pivot_table(
            self.data, 
            values='Survived', 
            index=['AgeGroup'], 
            columns=['Sex'], 
            aggfunc=np.mean
        ) * 100
        
        # Plot
        plt.figure(figsize=(14, 7))
        ax = pivot.plot(kind='bar', colormap='viridis', width=0.8)
        plt.title('Survival Rate by Age Group and Gender', fontsize=14)
        plt.xlabel('Age Group', fontsize=12)
        plt.ylabel('Survival Rate (%)', fontsize=12)
        plt.legend(title='Gender')
        plt.ylim(0, 100)
        
        # Add percentage labels
        for i, age_group in enumerate(pivot.index):
            for j, gender in enumerate(pivot.columns):
                if (age_group, gender) in grouped:
                    rate = grouped[(age_group, gender)]
                    count = counts[(age_group, gender)]
                    x_pos = i - 0.15 + j*0.3
                    plt.text(x_pos, rate + 2, f'{rate:.1f}%', ha='center')
                    plt.text(x_pos, rate - 8, f'n={count}', ha='center', size=8)
        
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def analyze_family_survival(self):
        """
        Analyze how family size affected survival rates.
        
        This method:
        - Examines survival rates by family size
        - Visualizes the relationship between family size and survival
        - Identifies whether family size was a significant factor in survival
        """
        if 'FamilySize' not in self.data.columns:
            print("Required column 'FamilySize' not found in the dataset.")
            return
        
        # Group by FamilySize
        grouped = self.data.groupby('FamilySize')['Survived'].mean() * 100
        counts = self.data.groupby('FamilySize').size()
        
        print("\n=== Survival Rate by Family Size ===")
        for family_size, rate in grouped.items():
            count = counts[family_size]
            print(f"- Family size {family_size}: {rate:.2f}% ({count} passengers)")
        
        # Add insights
        solo_rate = grouped.get(1, 0)
        small_family_rates = [grouped.get(size, 0) for size in [2, 3, 4]]
        avg_small_family = sum(small_family_rates) / len(small_family_rates) if small_family_rates else 0
        large_family_rates = [grouped.get(size, 0) for size in range(5, 12) if size in grouped.index]
        avg_large_family = sum(large_family_rates) / len(large_family_rates) if large_family_rates else 0
        
        if avg_small_family > solo_rate and avg_small_family > avg_large_family:
            self.add_insight(f"Small families (2-4 members) had the best survival rates ({avg_small_family:.1f}%), compared to solo travelers ({solo_rate:.1f}%) or large families ({avg_large_family:.1f}%)")
        
        if avg_large_family < solo_rate and avg_large_family < avg_small_family:
            self.add_insight(f"Large families (5+ members) had the worst survival rates ({avg_large_family:.1f}%)")
        
        # Plot
        plt.figure(figsize=(12, 6))
        
        plt.subplot(1, 2, 1)
        # Bar plot of survival rates
        bars = sns.barplot(x=grouped.index, y=grouped.values, palette='viridis')
        plt.title('Survival Rate by Family Size', fontsize=14)
        plt.xlabel('Family Size', fontsize=12)
        plt.ylabel('Survival Rate (%)', fontsize=12)
        plt.ylim(0, 100)
        
        # Add percentage labels
        for i, rate in enumerate(grouped.values):
            count = counts[grouped.index[i]]
            bars.text(i, rate + 3, f'{rate:.1f}%', ha='center')
            bars.text(i, rate - 8, f'n={count}', ha='center', size=8, color='white' if rate > 20 else 'black')
        
        plt.subplot(1, 2, 2)
        # Scatter plot with number of passengers as size
        sizes = counts.values * 5  # Scale for visibility
        plt.scatter(grouped.index, grouped.values, s=sizes, alpha=0.7, c=grouped.values, cmap='viridis')
        
        plt.title('Survival Rate vs Family Size', fontsize=14)
        plt.xlabel('Family Size', fontsize=12)
        plt.ylabel('Survival Rate (%)', fontsize=12)
        plt.ylim(0, 100)
        
        # Add trendline
        from scipy.stats import linregress
        slope, intercept, r_value, p_value, std_err = linregress(grouped.index, grouped.values)
        x_line = np.array([min(grouped.index), max(grouped.index)])
        y_line = intercept + slope * x_line
        plt.plot(x_line, y_line, color='red', linestyle='--')
        
        if p_value < 0.05:
            corr_strength = "strong" if abs(r_value) > 0.7 else "moderate" if abs(r_value) > 0.3 else "weak"
            direction = "positive" if slope > 0 else "negative"
            plt.text(0.05, 0.05, f"Correlation: {r_value:.2f}\n{corr_strength} {direction} relationship\np-value: {p_value:.3f}", 
                    transform=plt.gca().transAxes, fontsize=10, 
                    bbox=dict(facecolor='white', alpha=0.8))
        
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    def perform_eda(self):
        """
        Perform a comprehensive exploratory data analysis.
        
        This method:
        - Analyzes overall survival rates
        - Explores survival by key demographic features
        - Examines relationships between multiple variables
        - Identifies and compiles key insights
        """
        print("=== Exploratory Data Analysis ===\n")
        
        # Overall survival rate
        self.survival_rate_overview()
        
        # Key categorical variables
        for category in ['Sex', 'Pclass', 'Title', 'Embarked']:
            if category in self.data.columns:
                self.survival_by_category(category)
        
        # Key numeric variables
        for numeric_col in ['Age', 'Fare', 'FamilySize']:
            if numeric_col in self.data.columns:
                self.survival_by_numeric(numeric_col)
        
        # Correlation analysis
        self.correlation_analysis()
        
        # Combined analyses
        self.analyze_gender_class_survival()
        self.analyze_age_gender_survival()
        self.analyze_family_survival()
        
        # Summarize key insights
        print("\n=== Key Insights from Exploratory Analysis ===")
        for i, insight in enumerate(self.insights, 1):
            print(f"{i}. {insight}")

Now let's use the ExploratoryAnalysis class to explore our cleaned dataset:

In [ ]:
# Initialize and use the exploratory analysis
explorer = ExploratoryAnalysis(cleaned_df)
explorer.perform_eda()

## Statistical Analysis
Now, let's perform statistical testing to confirm our observations and quantify the significance of the patterns we've observed. We'll create a StatisticalAnalyzer class for this purpose.

In [ ]:
class StatisticalAnalyzer:
    """
    A class for performing statistical analysis on the Titanic dataset.
    
    This class provides methods to:
    - Perform chi-square tests for categorical variables
    - Perform t-tests for numeric variables
    - Analyze effect sizes to understand practical significance
    """
    
    def __init__(self, data):
        """
        Initialize with a pandas DataFrame.
        
        Parameters:
        -----------
        data : pandas.DataFrame
            The dataset to analyze
        """
        self.data = data
        self.statistical_findings = []  # Store key findings for later reporting
        
    def add_finding(self, finding):
        """Add a statistical finding to the collection for reporting."""
        self.statistical_findings.append(finding)
        print(f"✓ FINDING: {finding}")
        
    def chi_square_test(self, categorical_var):
        """
        Perform chi-square test of independence between categorical variable and survival.
        
        Parameters:
        -----------
        categorical_var : str
            Name of the categorical variable to test against survival
            
        This method:
        - Creates a contingency table
        - Performs the chi-square test
        - Calculates effect size using Cramer's V
        - Interprets and reports the findings
        """
        if 'Survived' not in self.data.columns:
            print("No 'Survived' column found in the dataset.")
            return
            
        if categorical_var not in self.data.columns:
            print(f"Variable '{categorical_var}' not found in the dataset.")
            return
        
        # Create contingency table
        contingency_table = pd.crosstab(self.data[categorical_var], self.data['Survived'])
        
        print(f"\n=== Chi-square Test: {categorical_var} vs Survived ===")
        print("Contingency Table:")
        display(contingency_table)
        
        # Calculate expected frequencies
        chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
        
        # Print results
        print(f"\nResults:")
        print(f"Chi-square value: {chi2:.4f}")
        print(f"p-value: {p:.8f}")
        print(f"Degrees of freedom: {dof}")
        
        # Interpret the result
        alpha = 0.05
        if p < alpha:
            significance = "Significant"
            print(f"Result: {significance} relationship between {categorical_var} and survival (p < {alpha})")
        else:
            significance = "No significant"
            print(f"Result: {significance} relationship between {categorical_var} and survival (p >= {alpha})")
            
        # Calculate Cramer's V for effect size
        n = contingency_table.sum().sum()
        cramer_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))
        print(f"Effect size (Cramer's V): {cramer_v:.4f}")
        
        # Interpret Cramer's V
        if cramer_v < 0.1:
            effect = "negligible"
        elif cramer_v < 0.2:
            effect = "weak"
        elif cramer_v < 0.3:
            effect = "moderate"
        else:
            effect = "strong"
            
        print(f"Interpretation: {effect.capitalize()} association")
        
        # Add finding
        if p < alpha:
            self.add_finding(f"{categorical_var} shows a statistically significant relationship with survival (p={p:.6f}), with a {effect} effect size")
            
            # Add category-specific findings
            if categorical_var == 'Sex':
                women_survival = contingency_table.loc['female', 1] / contingency_table.loc['female'].sum() * 100
                men_survival = contingency_table.loc['male', 1] / contingency_table.loc['male'].sum() * 100
                self.add_finding(f"Women had a {women_survival:.1f}% survival rate compared to {men_survival:.1f}% for men - this difference is statistically significant")
            
            elif categorical_var == 'Pclass' and 1 in contingency_table.index and 3 in contingency_table.index:
                first_survival = contingency_table.loc[1, 1] / contingency_table.loc[1].sum() * 100
                third_survival = contingency_table.loc[3, 1] / contingency_table.loc[3].sum() * 100
                self.add_finding(f"First class passengers had a {first_survival:.1f}% survival rate compared to {third_survival:.1f}% for third class - a statistically significant class difference")
    
    def t_test(self, numeric_var):
        """
        Perform t-test to compare means of numeric variable between survived and non-survived groups.
        
        Parameters:
        -----------
        numeric_var : str
            Name of the numeric variable to test
            
        This method:
        - Splits data by survival status
        - Performs independent two-sample t-test
        - Calculates Cohen's d for effect size
        - Interprets and reports the findings
        """
        if 'Survived' not in self.data.columns:
            print("No 'Survived' column found in the dataset.")
            return
            
        if numeric_var not in self.data.columns:
            print(f"Variable '{numeric_var}' not found in the dataset.")
            return
            
        if not np.issubdtype(self.data[numeric_var].dtype, np.number):
            print(f"Variable '{numeric_var}' is not numeric.")
            return
        
        # Split data by survival status
        survived = self.data[self.data['Survived'] == 1][numeric_var].dropna()
        not_survived = self.data[self.data['Survived'] == 0][numeric_var].dropna()
        
        # Calculate means
        survived_mean = survived.mean()
        not_survived_mean = not_survived.mean()
        
        print(f"\n=== T-test: {numeric_var} by Survival Status ===")
        print(f"Mean {numeric_var} by Survival Status:")
        print(f"- Survived: {survived_mean:.4f} (n={len(survived)})")
        print(f"- Not Survived: {not_survived_mean:.4f} (n={len(not_survived)})")
        print(f"- Difference: {survived_mean - not_survived_mean:.4f}")
        
        # Perform t-test
        t_stat, p_value = stats.ttest_ind(survived, not_survived, equal_var=False)
        
        # Print results
        print(f"\nResults:")
        print(f"t-statistic: {t_stat:.4f}")
        print(f"p-value: {p_value:.8f}")
        
        # Interpret the result
        alpha = 0.05
        if p_value < alpha:
            significance = "Significant"
            print(f"Result: {significance} difference in {numeric_var} between survivors and non-survivors (p < {alpha})")
        else:
            significance = "No significant"
            print(f"Result: {significance} difference in {numeric_var} between survivors and non-survivors (p >= {alpha})")
            
        # Calculate Cohen's d for effect size
        pooled_std = np.sqrt(((len(survived) - 1) * survived.std()**2 + 
                               (len(not_survived) - 1) * not_survived.std()**2) / 
                              (len(survived) + len(not_survived) - 2))
        cohens_d = abs(survived_mean - not_survived_mean) / pooled_std
        
        print(f"Effect size (Cohen's d): {cohens_d:.4f}")
        
        # Interpret Cohen's d
        if cohens_d < 0.2:
            effect = "negligible"
        elif cohens_d < 0.5:
            effect = "small"
        elif cohens_d < 0.8:
            effect = "medium"
        else:
            effect = "large"
            
        print(f"Interpretation: {effect.capitalize()} effect size")
        
        # Add finding
        if p_value < alpha:
            self.add_finding(f"The difference in {numeric_var} between survivors and non-survivors is statistically significant (p={p_value:.6f}), with a {effect} effect size")
            
            if numeric_var == 'Age':
                survivors_younger = survived_mean < not_survived_mean
                diff = abs(survived_mean - not_survived_mean)
                comparative = "younger" if survivors_younger else "older"
                self.add_finding(f"Survivors were {diff:.1f} years {comparative} on average than non-survivors")
            
            elif numeric_var == 'Fare':
                survivors_paid_more = survived_mean > not_survived_mean
                percent_diff = (abs(survived_mean - not_survived_mean) / not_survived_mean) * 100
                comparative = "more" if survivors_paid_more else "less"
                self.add_finding(f"Survivors paid {percent_diff:.1f}% {comparative} on average for their tickets (£{survived_mean:.2f} vs £{not_survived_mean:.2f})")
        
        # Visualize the distributions
        plt.figure(figsize=(10, 6))
        sns.histplot(survived, color='green', label='Survived', alpha=0.5, kde=True)
        sns.histplot(not_survived, color='red', label='Not Survived', alpha=0.5, kde=True)
        plt.axvline(survived_mean, color='green', linestyle='--', linewidth=2)
        plt.axvline(not_survived_mean, color='red', linestyle='--', linewidth=2)
        plt.title(f'Distribution of {numeric_var} by Survival Status', fontsize=14)
        plt.xlabel(numeric_var, fontsize=12)
        plt.ylabel('Count', fontsize=12)
        plt.legend()
        
        # Add annotation with statistics
        stats_text = f"t = {t_stat:.2f}, p = {p_value:.4f}\nCohen's d = {cohens_d:.2f} ({effect} effect)"
        plt.annotate(stats_text, xy=(0.05, 0.95), xycoords='axes fraction', 
                     fontsize=12, ha='left', va='top',
                     bbox=dict(boxstyle='round', alpha=0.1))
        
        plt.grid(alpha=0.3)
        plt.show()
    
    
    
    def perform_statistical_analysis(self):
        """
        Perform comprehensive statistical analysis.
        
        This method:
        - Conducts chi-square tests for categorical variables
        - Performs t-tests for numeric variables
        - Compiles and summarizes statistical findings
        """
        print("=== Statistical Analysis ===\n")
        
        # Chi-square tests for categorical variables
        for categorical_var in ['Sex', 'Pclass', 'Embarked', 'Title', 'FamilyType']:
            if categorical_var in self.data.columns:
                self.chi_square_test(categorical_var)
        
        # T-tests for numeric variables
        for numeric_var in ['Age', 'Fare', 'FamilySize']:
            if numeric_var in self.data.columns:
                self.t_test(numeric_var)
        
        
        # Summarize key findings
        print("\n=== Key Statistical Findings ===")
        for i, finding in enumerate(self.statistical_findings, 1):
            print(f"{i}. {finding}")

Let's use the StatisticalAnalyzer class to perform statistical analysis:

In [ ]:
# Initialize and use the statistical analyzer
analyzer = StatisticalAnalyzer(cleaned_df)
analyzer.perform_statistical_analysis()

## Generating a PDF Report
Now, let's create a class to generate a comprehensive PDF report of our analysis. This report will include our key findings, visualizations, and statistical results.

In [ ]:
class ReportGenerator:
    """
    A class for generating a PDF report of the Titanic dataset analysis.
    
    This class uses FPDF to create a report with:
    - A title page and introduction
    - Data overview section
    - Exploratory analysis results
    - Statistical findings
    - Key insights and conclusions
    - Visualizations from the analysis
    """
    
    def __init__(self, data, explorer, analyzer):
        """
        Initialize with the data and analysis results.
        
        Parameters:
        -----------
        data : pandas.DataFrame
            The cleaned dataset
        explorer : ExploratoryAnalysis
            The explorer object with EDA results
        analyzer : StatisticalAnalyzer
            The analyzer object with statistical results
        """
        self.data = data
        self.explorer = explorer
        self.analyzer = analyzer
        self.pdf = None
        
    def save_fig_for_pdf(self, plt_figure=None, filename=None, dpi=300):
        """
        Save the current matplotlib figure to a BytesIO object for PDF inclusion.
        
        Parameters:
        -----------
        plt_figure : matplotlib.figure.Figure, optional
            The figure to save. If None, uses current figure.
        filename : str, optional
            Filename for debugging (not used in the final PDF)
        dpi : int
            Resolution for the saved image
            
        Returns:
        --------
        BytesIO
            The image as a BytesIO object
        """
        img_data = BytesIO()
        if plt_figure:
            plt_figure.savefig(img_data, format='png', dpi=dpi, bbox_inches='tight')
        else:
            plt.savefig(img_data, format='png', dpi=dpi, bbox_inches='tight')
        img_data.seek(0)
        
        # For debugging, optionally save to file
        if filename:
            with open(filename, 'wb') as f:
                f.write(img_data.getbuffer())
        
        return img_data
    
    def initialize_pdf(self):
        """Initialize the PDF with basic settings."""
        self.pdf = FPDF()
        self.pdf.set_auto_page_break(auto=True, margin=15)
        self.pdf.add_page()
        
        # Add title
        self.pdf.set_font('Arial', 'B', 24)
        self.pdf.cell(0, 20, 'Titanic Disaster Analysis', ln=True, align='C')
        
        # Add subtitle
        self.pdf.set_font('Arial', 'I', 16)
        self.pdf.cell(0, 10, 'An Exploratory Data Analysis', ln=True, align='C')
        
        # Add date
        self.pdf.set_font('Arial', '', 12)
        today = datetime.now().strftime('%B %d, %Y')
        self.pdf.cell(0, 10, f'Generated on {today}', ln=True, align='C')
        
        # Add separator
        self.pdf.ln(10)
        self.pdf.line(10, self.pdf.get_y(), 200, self.pdf.get_y())
        self.pdf.ln(15)
    
    def add_section_header(self, title):
        """Add a section header to the PDF."""
        self.pdf.add_page()
        self.pdf.set_font('Arial', 'B', 18)
        self.pdf.cell(0, 10, title, ln=True)
        self.pdf.line(10, self.pdf.get_y(), 200, self.pdf.get_y())
        self.pdf.ln(10)
    
    def add_subsection_header(self, title):
        """Add a subsection header to the PDF."""
        self.pdf.set_font('Arial', 'B', 14)
        self.pdf.cell(0, 10, title, ln=True)
        self.pdf.ln(5)
    
    def add_paragraph(self, text):
        """Add a paragraph of text to the PDF."""
        self.pdf.set_font('Arial', '', 12)
        self.pdf.multi_cell(0, 6, text)
        self.pdf.ln(5)
    
    def add_bullet_point(self, text):
        """Add a bullet point to the PDF."""
        self.pdf.set_font('Helvetica', '', 12)
        self.pdf.multi_cell(0, 6, f"- {text}")
        self.pdf.ln(1)
    
    def add_image(self, img_data, w=150, h=75, caption=None):
        """
        Add an image to the PDF with proper page-break and cursor management.
        """
       
        caption_height = 12 if caption else 0
        total_height_needed = h + caption_height + 10
        
        
        if self.pdf.get_y() + total_height_needed > 270:
            self.pdf.add_page()
            
        import uuid
        import os
        img_id = str(uuid.uuid4())
        temp_img = f'temp_{img_id}.png'
        
        
        with open(temp_img, 'wb') as f:
            f.write(img_data.getbuffer())
        
        
        x = (self.pdf.w - w) / 2
        y = self.pdf.get_y()
        
        
        self.pdf.image(temp_img, x=x, y=y, w=w, h=h)
        
        
        if os.path.exists(temp_img):
            os.remove(temp_img)
        
        
        self.pdf.set_y(y + h + 2)
        
       
        if caption:
            self.pdf.set_font('Helvetica', 'I', 10)
            self.pdf.cell(0, 6, caption, align='C', new_x="LMARGIN", new_y="NEXT")
            self.pdf.ln(4)
        else:
            self.pdf.ln(5)
    
    def add_visualization(self, visualization_function, *args, **kwargs):
        """
        Add a visualization to the PDF.
        
        Parameters:
        -----------
        visualization_function : function
            The function to call to create the visualization
        *args, **kwargs
            Arguments to pass to the visualization function
        """
        # Create a new figure
        plt.figure(figsize=(10, 6))
        
        # Call the visualization function
        visualization_function(*args, **kwargs)
        
        # Save the figure to a BytesIO object
        img_data = self.save_fig_for_pdf()
        
        # Close the figure to avoid displaying it in the notebook
        plt.close()
        
        # Add the image to the PDF
        self.add_image(img_data)
    
    def add_data_overview(self):
        """Add data overview section to the PDF."""
        self.add_section_header("1. Data Overview")
        
        # Basic dataset info
        self.add_paragraph(f"The Titanic dataset contains information about {len(self.data)} passengers who were aboard the Titanic when it sank in 1912. The data includes demographic information, ticket details, cabin information, and survival status.")
        
        # Dataset statistics
        self.add_subsection_header("Dataset Statistics")
        self.add_paragraph("Key statistics about the Titanic passengers:")
        
        # Add some key stats as bullet points
        survival_rate = self.data['Survived'].mean() * 100
        self.add_bullet_point(f"Survival Rate: {survival_rate:.1f}%")
        
        male_percent = (self.data['Sex'] == 'male').mean() * 100
        self.add_bullet_point(f"Gender Distribution: {male_percent:.1f}% male, {100-male_percent:.1f}% female")
        
        class_dist = self.data['Pclass'].value_counts(normalize=True) * 100
        self.add_bullet_point(f"Class Distribution: {class_dist[1]:.1f}% first class, {class_dist[2]:.1f}% second class, {class_dist[3]:.1f}% third class")
        
        age_mean = self.data['Age'].mean()
        age_min = self.data['Age'].min()
        age_max = self.data['Age'].max()
        self.add_bullet_point(f"Age Range: {age_min:.0f} to {age_max:.0f} years (mean: {age_mean:.1f})")
        
        # Add a visual representation of the dataset
        self.add_subsection_header("Passenger Class Distribution")
        
        # Create class distribution visualization
        plt.figure(figsize=(8, 5))
        class_counts = self.data['Pclass'].value_counts().sort_index()
        sns.barplot(x=class_counts.index, y=class_counts.values, palette='viridis')
        plt.title('Distribution by Passenger Class')
        plt.xlabel('Passenger Class')
        plt.ylabel('Count')
        plt.xticks([0, 1, 2], ['1st', '2nd', '3rd'])
        
        # Add percentage labels
        for i, count in enumerate(class_counts):
            percentage = count / len(self.data) * 100
            plt.text(i, count + 5, f'{percentage:.1f}%', ha='center')
        
        plt.grid(axis='y', alpha=0.3)
        
        # Save and add the figure
        img_data = self.save_fig_for_pdf()
        plt.close()
        self.add_image(img_data, caption="Figure 1: Distribution of passengers by class")
        
        # Add information about missing values
        self.add_subsection_header("Missing Values")
        missing_values = self.data.isnull().sum()
        missing_values = missing_values[missing_values > 0]
        
        if len(missing_values) > 0:
            self.add_paragraph("The following variables had missing values in the original dataset:")
            for col, count in missing_values.items():
                percentage = count / len(self.data) * 100
                self.add_bullet_point(f"{col}: {count} missing values ({percentage:.1f}%)")
        else:
            self.add_paragraph("All missing values have been addressed in the data cleaning process.")
    
    def add_survival_analysis(self):
        """Add survival analysis section to the PDF."""
        self.add_section_header("2. Survival Analysis")
        
        # Overall survival
        self.add_subsection_header("Overall Survival Rate")
        survival_rate = self.data['Survived'].mean() * 100
        self.add_paragraph(f"Of the {len(self.data)} passengers in the dataset, {self.data['Survived'].sum()} survived the disaster, representing a survival rate of {survival_rate:.1f}%.")
        
        # Create overall survival chart
        plt.figure(figsize=(8, 5))
        sns.countplot(x='Survived', data=self.data, palette=['#FF5555', '#55AA55'])
        plt.title('Survival Distribution')
        plt.xlabel('Survived (0 = No, 1 = Yes)')
        plt.ylabel('Count')
        plt.xticks([0, 1], ['No', 'Yes'])
        
        # Add percentage labels
        for i, count in enumerate(self.data['Survived'].value_counts().sort_index()):
            percentage = count / len(self.data) * 100
            plt.text(i, count + 15, f'{percentage:.1f}%', ha='center')
        
        plt.grid(axis='y', alpha=0.3)
        
        # Save and add the figure
        img_data = self.save_fig_for_pdf()
        plt.close()
        self.add_image(img_data, caption="Figure 2: Overall survival distribution")
        
        # Survival by gender
        self.add_subsection_header("Survival by Gender")
        self.add_paragraph("One of the most striking patterns in the data is the difference in survival rates between men and women.")
        
        # Create gender survival chart
        gender_survival = self.data.groupby('Sex')['Survived'].mean() * 100
        
        plt.figure(figsize=(8, 5))
        sns.barplot(x=gender_survival.index, y=gender_survival.values, palette='viridis')
        plt.title('Survival Rate by Gender')
        plt.xlabel('Gender')
        plt.ylabel('Survival Rate (%)')
        plt.ylim(0, 100)
        
        # Add percentage labels
        for i, rate in enumerate(gender_survival):
            plt.text(i, rate + 3, f'{rate:.1f}%', ha='center')
        
        plt.grid(axis='y', alpha=0.3)
        
        # Save and add the figure
        img_data = self.save_fig_for_pdf()
        plt.close()
        self.add_image(img_data, caption="Figure 3: Survival rate by gender")
        
        female_rate = gender_survival['female']
        male_rate = gender_survival['male']
        self.add_paragraph(f"Women had a survival rate of {female_rate:.1f}%, compared to only {male_rate:.1f}% for men. This supports the historical accounts of 'women and children first' evacuation policies on the Titanic.")
        
        # Survival by class
        self.add_subsection_header("Survival by Passenger Class")
        self.add_paragraph("Passenger class was another significant factor affecting survival chances.")
        
        # Create class survival chart
        class_survival = self.data.groupby('Pclass')['Survived'].mean() * 100
        
        plt.figure(figsize=(8, 5))
        sns.barplot(x=class_survival.index, y=class_survival.values, palette='viridis')
        plt.title('Survival Rate by Passenger Class')
        plt.xlabel('Passenger Class')
        plt.ylabel('Survival Rate (%)')
        plt.xticks([0, 1, 2], ['1st', '2nd', '3rd'])
        plt.ylim(0, 100)
        
        # Add percentage labels
        for i, rate in enumerate(class_survival):
            plt.text(i, rate + 3, f'{rate:.1f}%', ha='center')
        
        plt.grid(axis='y', alpha=0.3)
        
        # Save and add the figure
        img_data = self.save_fig_for_pdf()
        plt.close()
        self.add_image(img_data, caption="Figure 4: Survival rate by passenger class")
        
        first_rate = class_survival[1]
        third_rate = class_survival[3]
        self.add_paragraph(f"First-class passengers had a survival rate of {first_rate:.1f}%, compared to only {third_rate:.1f}% for third-class passengers. This significant disparity suggests that access to lifeboats was not equal across passenger classes.")
        
        # Survival by age
        self.add_subsection_header("Survival by Age")
        self.add_paragraph("Age also played an important role in determining survival chances.")
        
        # Create age group survival chart if AgeGroup exists
        if 'AgeGroup' in self.data.columns:
            age_group_survival = self.data.groupby('AgeGroup')['Survived'].mean() * 100
            
            plt.figure(figsize=(10, 5))
            sns.barplot(x=age_group_survival.index, y=age_group_survival.values, palette='viridis')
            plt.title('Survival Rate by Age Group')
            plt.xlabel('Age Group')
            plt.ylabel('Survival Rate (%)')
            plt.ylim(0, 100)
            plt.xticks(rotation=45)
            
            # Add percentage labels
            for i, rate in enumerate(age_group_survival):
                plt.text(i, rate + 3, f'{rate:.1f}%', ha='center')
            
            plt.grid(axis='y', alpha=0.3)
            plt.tight_layout()
            
            # Save and add the figure
            img_data = self.save_fig_for_pdf()
            plt.close()
            self.add_image(img_data, caption="Figure 5: Survival rate by age group")
            
            if 'Child' in age_group_survival.index:
                child_rate = age_group_survival['Child']
                self.add_paragraph(f"Children under 12 years old had a survival rate of {child_rate:.1f}%, supporting the 'women and children first' evacuation policy.")
        
        # Combined effects of gender and class
        self.add_subsection_header("Combined Effects of Gender and Class")
        self.add_paragraph("When we examine the combined effects of gender and class, we see even more striking patterns:")
        
        # Create a pivot table for plotting
        pivot = pd.pivot_table(
            self.data, 
            values='Survived', 
            index=['Sex'], 
            columns=['Pclass'], 
            aggfunc=np.mean
        ) * 100
        
        # Plot
        plt.figure(figsize=(10, 6))
        pivot.plot(kind='bar', colormap='viridis')
        plt.title('Survival Rate by Gender and Class')
        plt.xlabel('Gender')
        plt.ylabel('Survival Rate (%)')
        plt.xticks(ticks=[0, 1], labels=['Female', 'Male'], rotation=0)
        plt.legend(title='Passenger Class')
        plt.ylim(0, 100)
        
        # Add percentage labels
        for i, (sex, row) in enumerate(pivot.iterrows()):
            for j, rate in enumerate(row):
                plt.text(i - 0.25 + j*0.25, rate + 2, f'{rate:.1f}%', ha='center')
        
        plt.grid(axis='y', alpha=0.3)
        
        # Save and add the figure
        img_data = self.save_fig_for_pdf()
        plt.close()
        self.add_image(img_data, caption="Figure 6: Survival rate by gender and class")
        
        female_first = pivot.loc['female', 1] if ('female', 1) in pivot.index else 0
        male_third = pivot.loc['male', 3] if ('male', 3) in pivot.index else 0
        self.add_paragraph(f"The highest survival rate was among first-class women ({female_first:.1f}%), while third-class men had the lowest survival rate ({male_third:.1f}%). This dramatic difference highlights the combined effects of gender and social class during the disaster.")
    
    def add_statistical_findings(self):
        """Add statistical findings section to the PDF."""
        self.add_section_header("3. Statistical Analysis")
        
        self.add_paragraph("To validate our observations from exploratory analysis, we conducted several statistical tests to determine which factors had statistically significant relationships with survival.")
        
        # Chi-square test results
        self.add_subsection_header("Chi-Square Tests for Categorical Variables")
        self.add_paragraph("Chi-square tests were used to determine if categorical variables like Sex, Pclass, and Title had significant relationships with survival:")
        
        for i, finding in enumerate(self.analyzer.statistical_findings, 1):
            if "statistically significant" in finding and "effect size" in finding:
                self.add_bullet_point(finding)
        
        # T-test results
        self.add_subsection_header("T-Tests for Numeric Variables")
        self.add_paragraph("Independent t-tests were conducted to compare differences in numeric variables between survivors and non-survivors:")
        
        for i, finding in enumerate(self.analyzer.statistical_findings, 1):
            if "difference" in finding and "statistically significant" in finding:
                self.add_bullet_point(finding)
        
        # Create a visualization showing age distribution by survival
        if 'Age' in self.data.columns:
            plt.figure(figsize=(10, 6))
            survived = self.data[self.data['Survived'] == 1]['Age'].dropna()
            not_survived = self.data[self.data['Survived'] == 0]['Age'].dropna()
            
            sns.histplot(survived, color='green', label='Survived', alpha=0.5, kde=True)
            sns.histplot(not_survived, color='red', label='Not Survived', alpha=0.5, kde=True)
            plt.axvline(survived.mean(), color='green', linestyle='--', linewidth=2)
            plt.axvline(not_survived.mean(), color='red', linestyle='--', linewidth=2)
            plt.title('Distribution of Age by Survival Status')
            plt.xlabel('Age')
            plt.ylabel('Count')
            plt.legend()
            
            # Add annotation with statistics
            t_stat, p_value = stats.ttest_ind(survived, not_survived, equal_var=False)
            stats_text = f"t = {t_stat:.2f}, p = {p_value:.4f}"
            plt.annotate(stats_text, xy=(0.05, 0.95), xycoords='axes fraction', 
                         fontsize=12, ha='left', va='top',
                         bbox=dict(boxstyle='round', alpha=0.1))
            
            # Save and add the figure
            img_data = self.save_fig_for_pdf()
            plt.close()
            self.add_image(img_data, caption="Figure 7: Age distribution by survival status")
    
    def add_key_insights(self):
        """Add key insights section to the PDF."""
        self.add_section_header("4. Key Insights and Conclusions")
        
        self.add_paragraph("Based on our exploratory and statistical analysis of the Titanic dataset, we can draw several important conclusions about the factors that influenced survival:")
        
        # Gender effects
        self.add_subsection_header("Gender and Survival")
        self.add_paragraph("Gender was the most significant factor influencing survival chances. Women were much more likely to survive than men, which aligns with the 'women and children first' maritime evacuation protocol. This gender disparity persisted across all passenger classes, though it was less pronounced in third class.")
        
        # Class effects
        self.add_subsection_header("Social Class and Survival")
        self.add_paragraph("Passenger class showed a strong relationship with survival probability. First-class passengers had significantly higher survival rates than third-class passengers. This may be due to several factors:")
        self.add_bullet_point("First-class cabins were closer to the boat deck where lifeboats were located")
        self.add_bullet_point("First-class passengers may have been given priority access to lifeboats")
        self.add_bullet_point("Third-class passengers may have received delayed notifications about the emergency")
        
        # Age effects
        self.add_subsection_header("Age and Survival")
        self.add_paragraph("Age was also a significant factor in survival, with children having higher survival rates than adults. This supports historical accounts that children were prioritized during evacuation, along with women.")
        
        # Family effects
        if 'FamilySize' in self.data.columns:
            self.add_subsection_header("Family Size and Survival")
            family_survival = self.data.groupby('FamilySize')['Survived'].mean()
            
            # Check if we have enough data to make this conclusion
            solo_rate = family_survival.get(1, 0)
            small_family_rates = [family_survival.get(size, 0) for size in [2, 3, 4]]
            avg_small_family = sum(small_family_rates) / len(small_family_rates) if small_family_rates else 0
            
            if avg_small_family > solo_rate:
                self.add_paragraph(f"Passengers traveling in small family groups (2-4 members) generally had better survival rates ({avg_small_family*100:.1f}%) than solo travelers ({solo_rate*100:.1f}%). This suggests that families might have been kept together during evacuation, or that family members helped each other reach safety.")
            else:
                self.add_paragraph("Family size showed some relationship with survival, though the pattern was not as clear as with gender and class.")
        
        # Overall conclusions
        self.add_subsection_header("Final Conclusions")
        self.add_paragraph("The Titanic disaster reflected social hierarchies and gender norms of the early 20th century. The data strongly supports that women and children were prioritized for rescue, but also that social class played a significant role in determining who survived.")
        
        self.add_paragraph("The combined effects of these factors created dramatically different survival chances. For example, first-class women had survival rates above 90%, while third-class men had rates below 20%. These disparities highlight how social and demographic factors influenced life-or-death outcomes during this historic maritime disaster.")
    
    def generate_report(self, filename='Titanic_Analysis_Report.pdf'):
        """
        Generate the complete PDF report.
        
        Parameters:
        -----------
        filename : str
            Name of the output PDF file
            
        Returns:
        --------
        str
            Path to the generated PDF file
        """
        # Initialize the PDF
        self.initialize_pdf()
        
        # Add introduction
        self.add_paragraph("This report presents an analysis of the Titanic disaster dataset, examining the factors that influenced passenger survival rates. Using exploratory data analysis and statistical tests, we investigate how variables such as gender, age, passenger class, and family relationships affected chances of survival.")
        
        # Add main sections
        self.add_data_overview()
        self.add_survival_analysis()
        self.add_statistical_findings()
        self.add_key_insights()
        
        # Add closing
        self.pdf.add_page()
        self.pdf.set_font('Arial', 'B', 14)
        self.pdf.cell(0, 10, "Thank You", ln=True, align='C')
        
        self.pdf.set_font('Arial', '', 12)
        self.pdf.multi_cell(0, 6, "This analysis was prepared using Python with pandas, numpy, matplotlib, seaborn, and scipy libraries. The report was automatically generated using the FPDF library.")
        
        # Output the PDF
        self.pdf.output(filename)
        print(f"Report generated successfully: {filename}")
        return filename

Now let's use the ReportGenerator class to create a comprehensive PDF report of our analysis:

In [ ]:
print("explorer exists:", 'explorer' in globals())
print("analyzer exists:", 'analyzer' in globals())

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

In [ ]:
# Initialize the report generator
report_gen = ReportGenerator(cleaned_df, explorer, analyzer)

# Generate the report
report_file = report_gen.generate_report('Titanic_Analysis_Report.pdf')
print("Report generated successfully:", report_file)

In [ ]:
import os

current_dir = os.getcwd()
pdf_path = os.path.abspath('Titanic_Analysis_Report.pdf')

print("Direction:", current_dir)
print("Full_path:", pdf_path)
print("It_exists?", os.path.exists(pdf_path))

if os.path.exists(pdf_path):
    os.system(f'explorer /select,"{pdf_path}"')
else:
    os.system(f'explorer "{current_dir}"')

## Conclusion
In this comprehensive analysis of the Titanic dataset, we have:
1. **Loaded and explored the raw data** - Using our `DataLoader` class, we examined the structure and content of the dataset, identifying missing values and understanding the distribution of key variables.
2. **Cleaned and preprocessed the data** - Our `DataCleaner` class handled missing values, identified outliers, and created new features that enhanced our analysis.
3. **Performed exploratory data analysis** - The `ExploratoryAnalysis` class helped us uncover patterns in the data, particularly how factors like gender, age, and passenger class influenced survival rates.
4. **Conducted statistical tests** - Using the `StatisticalAnalyzer` class, we validated our observations with formal statistical tests, confirming which factors had statistically significant relationships with survival.
5. **Generated a comprehensive report** - The `ReportGenerator` class compiled our findings into a professional PDF report that documents the entire analysis process and key insights.
### Key Findings
Our analysis revealed several important factors that influenced survival rates on the Titanic:
- **Gender was the most significant factor** - Women had much higher survival rates than men, supporting the "women and children first" evacuation policy.
- **Social class played a crucial role** - First-class passengers had significantly higher survival rates than third-class passengers, highlighting the impact of social stratification on survival chances.
- **Age affected survival chances** - Children had higher survival rates than adults, though this effect varied by gender and class.
- **Family relationships mattered** - Passengers traveling in small family groups generally had better survival rates than solo travelers.
These findings provide valuable insights into how social, demographic, and situational factors influenced survival during one of history's most famous maritime disasters.
The object-oriented approach used in this analysis made the code more organized, reusable, and maintainable. Each class had a specific responsibility, making the code easier to understand and modify. This approach could be applied to other datasets and analytical problems as well.